## Lab-11 Model Inference on MQTT topic, Inference using REST-API
*Suggested time: 40-45 minutes*

Accelerator: CPU

In this lab we learn how to communicate model inference using MQTT protocol and API request.  We start by providing some exemplar code to demonstrate,
- Publish and subsribe to a MQTT topic (step 2).
- Step 3 of the notebook show examples of communicating inference of linear model and CPU utilization
- The last part of the notebook demonstrates use of Flask server to communicate inference from ResNet50 model by responding to an API request

##### **Step 0:**

Set up access to terminal on Colab machine and download sample model files and images.

In [22]:
!wget -qq https://edge-ai-doulos.s3.us-west-2.amazonaws.com/Edge-AI-Upload.zip
!unzip -qq Edge-AI-Upload.zip

##### **Step 1:**
    
Install Python MQTT Client - Paho  (pip install paho-mqtt) and tflite runtime

In [0]:
!pip install -q paho-mqtt==1.6.1
!pip install -q ai-edge-litert

##### **Step 2:**

Publish an incrementing counter value to a topic "city/street/car".
Subscribe to the same topic to see the published value.

*You can stop the execution of the cell after a few outputs.*


In [21]:
import paho.mqtt.client as mqtt
import time


def on_connect(client, userdata, flags, rc):
    print("Connected with result code " + str(rc))


def on_message(client, userdata, msg):
    print(msg.topic + " " + str(msg.payload))


# MQTT configuration (DRY)
TOPIC = "city/street/cars"
BROKER_ADDRESS = "broker.hivemq.com"
BROKER_PORT = 1883
BROKER_TIMEOUT = 60
PUBLISH_INTERVAL = 2
MAX_ITERATIONS = 5

# Last will testament (LWT) details
lwt_message = "The traffic camera is offline"
lwt_qos = 0  # QoS 0, 1, or 2
lwt_retain = False  # Set to True if you want the message retained

client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

try:
    # 1883 is the port number, while 60 seconds is the timeout
    client.connect(BROKER_ADDRESS, BROKER_PORT, BROKER_TIMEOUT)
    client.loop_start()

    # Subscribe before publishing so on_message can receive messages from the broker
    client.subscribe(TOPIC)

    counter = 0

    # Please customize the topic to - city/street/cars/toyota
    while counter < MAX_ITERATIONS:
        result = client.publish(TOPIC, counter)
        result.wait_for_publish()
        time.sleep(PUBLISH_INTERVAL)
        counter += 1

except OSError as e:
    print(f"Network error during MQTT connection or loop start: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Ensure client loop is stopped and disconnected
    client.loop_stop()
    client.disconnect()
    print("MQTT client disconnected.")


Connected with result code 0
city/street/cars b'0'
city/street/cars b'1'


city/street/cars b'2'


city/street/cars b'3'


city/street/cars b'4'


MQTT client disconnected.


KeyboardInterrupt: 

In [29]:
import paho.mqtt.client as mqtt
import time
import ssl # Import the ssl module

MQTT_BROKER = {
    "address": "broker.hivemq.com",
    "port": 8883,  # Standard port for MQTT over TLS
    "username": "rahuldee",
    "password": "rahul-hivemq",
    # Path to the broker's CA certificate (optional but highly recommended for validation)
    # "ca_cert_path": "path/to/ca.crt",
}

client.username_pw_set(MQTT_BROKER["username"], MQTT_BROKER["password"])

def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Connected successfully to the secure broker!")
        # Subscribing inside on_connect means if the connection is lost
        # the subscription will be renewed automatically upon reconnecting.
        client.subscribe("city/street/cars")
        print("Subscribed to topic: city/street/cars")
    else:
        print(f"Failed to connect with code: {rc}")

def on_message(client, userdata, msg):
    print(f"Received message on {msg.topic}: {msg.payload.decode()}")

def main():
    # Initialize the client
    client = mqtt.Client()

    # Set username and password
    client.username_pw_set(USERNAME, PASSWORD)

    # Configure TLS/SSL encryption
    # cert_reqs=ssl.CERT_REQUIRED verifies that the broker presents a valid certificate
    # If you want encrypted MQTT, uncomment the next line and provide a valid CA cert path.
    client.tls_set(tls_version=ssl.PROTOCOL_TLS) # Enabled TLS for secure connection

    # Assign event callbacks
    client.on_connect = on_connect
    client.on_message = on_message

    # Connect to the broker
    try:
        print(f"Attempting to connect to {MQTT_BROKER['address']}:{MQTT_BROKER['port']}")
        client.connect(MQTT_BROKER["address"], MQTT_BROKER["port"], keepalive=60)

        # Start the network loop in a background thread
        client.loop_start()

        # Give some time for connection and subscription to complete
        time.sleep(5) # Increased sleep time

        print("Publishing message...")
        client.publish("city/street/cars", "Hello from Jupyter Secure")

        # Keep the loop running to receive potential messages after publishing
        # For demonstration, we'll keep it running for a bit longer.
        # In a real application, you might use loop_forever() or a more sophisticated stopping mechanism.
        time.sleep(10) # Keep listening for a total of 10 seconds after publish

        print("Disconnecting MQTT client.")

    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        # Ensure the loop is stopped and client is disconnected
        client.loop_stop()
        client.disconnect()

if __name__ == "__main__":
    main()


Attempting to connect to broker.hivemq.com:8883


Connected successfully to the secure broker!
Subscribed to topic: city/street/cars


Publishing message...
Received message on city/street/cars: Hello from Jupyter Secure


KeyboardInterrupt: 

##### **Step 3(a)**

Publish model inference to a MQTT topic called *linear/inference*

*You can stop the execution of the cell after a few outputs.*

In [25]:
import paho.mqtt.client as mqtt  #import MQTT client functionality
import time
from ai_edge_litert.interpreter import Interpreter
import numpy as np


interpreter = Interpreter(model_path=str('Model-Files/linear.tflite'))
interpreter.allocate_tensors()
n_steps_in = 1
input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

def inference(input_inference):
    x_input=np.array([input_inference])
    x_input = x_input.reshape((1, n_steps_in))
    input_data = np.array(x_input, dtype=np.float32)
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    # The function `get_tensor()` returns a copy of the tensor data.
    prediction = interpreter.get_tensor(output_index)
    #print (type([prediction]))
    #Use item to convert numpy array value to Python datatype
    predicted_value = prediction[0][0]
    linear_string = str(predicted_value)
    return linear_string

def on_connect(client, userdata, flags, rc):
    print("connection received with result code " + str(rc))

def on_message(client1, userdata, message):
    print("message received ", str(message.payload.decode("utf-8")))

# Broker configuration for MQTT communications (DRY)
broker_addresses = {
    "default": "broker.hivemq.com",
    # If needed, replace the default with another broker once here
    # "fallback": "test.mosquitto.org",
}
broker_address = broker_addresses["default"]

# Topics used throughout the MQTT workflow
mqtt_topics = {
    "inference": "linear/inference",
}

# Create new client instance
client = mqtt.Client()

# Attach callbacks
client.on_connect = on_connect
client.on_message = on_message

# Connect to broker. Broker can be located on edge or cloud
client.connect(broker_address)

# Start the loop to process receive and send buffers
client.loop_start()
input_inference = 10

while True:
    print(f'Input for inference is {input_inference}')
    linear_inference = inference(input_inference)
    client.publish(mqtt_topics["inference"], linear_inference)
    time.sleep(5)  # Sleep for 5 seconds before next inference
    client.subscribe(mqtt_topics["inference"])
    input_inference += 10

client.disconnect()
client.loop_stop()

Input for inference is 10


connection received with result code 0


Input for inference is 20
message received  35.29982


Input for inference is 30
message received  53.11036


KeyboardInterrupt: 

##### **Observation & Exercise**

You will observe that CPU utilization goes very high when executing MQTT code in Jupyter Notebook environment.
Save this code using %%writefile command and then execute this code as a Python script in the bash shell and observe CPU utilization

##### **Step 3(b)**

Code below calculates CPU utilisation using data from */proc/stat*.

*You can stop the execution of the cell after a few outputs.*

**Exercise:** Publish CPU utilization to a topic of your choice.  Remotely monitor CPU utilization by subscribing to same topic.

In [0]:
from time import sleep


last_idle = last_total = 0
while True:
    with open('/proc/stat') as f:
        fields = [float(column) for column in f.readline().strip().split()[1:]]
    idle, total = fields[3], sum(fields)
    idle_delta, total_delta = idle - last_idle, total - last_total
    last_idle, last_total = idle, total
    utilisation = 100.0 * (1.0 - idle_delta / total_delta)
    print(f'CPU utilization is {utilisation:5.1f}')
    sleep(5)

#### **The code in the next two cells do not run in the Notebook.**

*Execute it on command line terminal using the instructions mentioned in Step 4*

REST API - Start Flask Server

Access ResNet50 Model inference using Pythons's flask web server module and REST API.




In [0]:
%%writefile run_keras_server.py

# USAGE
# Start the server:
# 	python run_keras_server.py
# Submit a request via cURL:
# 	curl -X POST -F image=@dog.jpg 'http://localhost:5000/predict'
# Submits a request via Python:
#	python simple_request.py

# import the necessary packages

from tensorflow import keras
from keras.applications import ResNet50
from keras.utils import img_to_array
from keras.applications import imagenet_utils
from PIL import Image
import numpy as np
import flask
import io

# initialize our Flask application and the Keras model
app = flask.Flask(__name__)

model = None

def load_model():
	# load the pre-trained Keras model (here we are using a model
	# pre-trained on ImageNet and provided by Keras, but you can
	# substitute in your own networks just as easily)
	global model
	model= ResNet50(weights='imagenet')


def prepare_image(image, target):
	# if the image mode is not RGB, convert it
	if image.mode != "RGB":
		image = image.convert("RGB")

	# resize the input image and preprocess it
	image = image.resize(target)
	image = img_to_array(image)
	image = np.expand_dims(image, axis=0)
	image = imagenet_utils.preprocess_input(image)

	# return the processed image
	return image

@app.route("/predict", methods=["POST"])
def predict():
	# initialize the data dictionary that will be returned from the
	# view
	data = {"success": False}

	# ensure an image was properly uploaded to our endpoint
	if flask.request.method == "POST":
		if flask.request.files.get("image"):
			# read the image in PIL format
			image = flask.request.files["image"].read()
			image = Image.open(io.BytesIO(image))

			# preprocess the image and prepare it for classification
			image = prepare_image(image, target=(224, 224))

			# classify the input image and then initialize the list
			# of predictions to return to the client
			preds = model.predict(image)
			results = imagenet_utils.decode_predictions(preds)
			data["predictions"] = []

			# loop over the results and add them to the list of
			# returned predictions
			for (imagenetID, label, prob) in results[0]:
				r = {"label": label, "probability": float(prob)}
				data["predictions"].append(r)

			# indicate that the request was a success
			data["success"] = True

	# return the data dictionary as a JSON response
	return flask.jsonify(data)

# if this is the main thread of execution first load the model and
# then start the server
if __name__ == "__main__":
	print(("* Loading Keras model and Flask starting server..."
		"please wait until server has fully started"))
	load_model()
	app.run()

In [0]:
%%writefile simple_request.py

# USAGE
# python simple_request.py

# import the necessary packages
import requests

# initialize the Keras REST API endpoint URL along with the input
# image path
KERAS_REST_API_URL = "http://localhost:5000/predict"
IMAGE_PATH = "Images/tiger.jpg"

# load the input image and construct the payload for the request
image = open(IMAGE_PATH, "rb").read()
payload = {"image": image}

# submit the request
r = requests.post(KERAS_REST_API_URL, files=payload).json()

# ensure the request was sucessful
if r["success"]:
	# loop over the predictions and display them
	for (i, result) in enumerate(r["predictions"]):
		print("{}. {}: {:.4f}".format(i + 1, result["label"],
			result["probability"]))

# otherwise, the request failed
else:
	print("Request failed")

##### **Step 4:** Start the Flask Server using subprocess Popen


In [0]:
import subprocess

# This launches the server and keeps it running in the background
with open("server_log.txt", "w") as log_file:
    process = subprocess.Popen(
        ["python", "run_keras_server.py"],
        stdout=log_file,
        stderr=log_file,
        preexec_fn=None # Use os.setpgrp if you want it to persist after cell stop
    )

print(f"Server started with PID: {process.pid}")

##### **Step 5: Send request to Flask request**

Now send a API request to the flask server by opening a new terminal and running the Python code seen below (simple_request.py)

You should see the following output, when a request to identify a picture of dog is sent to flask server.

1. tiger: 0.9192
2. tiger_cat: 0.0748
3. jaguar: 0.0047
4. zebra: 0.0010
5. leopard: 0.0001

In [0]:
!python simple_request.py

#### Questions:

A. Purpose of Last Will and Testament(LWT) in MQTT?

B. How is LWT implemented using Paho?

### Answer

<details>
    <summary> Click here to view our answer </summary>

    -  It offers clients to respond to ungraceful disconnects.

    - Start with creating LWT details. Then use the **will_set** method to activate it.

         lwt_topic = "city/street/cars

          lwt_message = "The traffic camera is offline"

          lwt_qos = 0

          lwt_retain = False  # Set to True if you want the message retained

          client = mqtt.Client()

          client.will_set(lwt_topic, lwt_message, lwt_qos, lwt_retain) 

    
</details>

  

#### **Demonstation on updating a topic using inference from an edge AI model**

Remotely monitor a MQTT topic populated by the Demo code running an object detection model on the Raspberry Pi. You should see changes in the topic whenever the demo code detects a clock. Topic name is ***clock/detected***

In [0]:
import paho.mqtt.client as mqtt
import datetime
import time

import paho.mqtt.subscribe as subscribe

def on_connect(client, userdata, flags, rc):
    print("Connected with result code " + str(rc))
    client.subscribe("clock/detected")


def on_message(client, userdata, msg):
    print(msg.topic + " " + str(msg.payload.decode("utf-8")))


# Broker for prototyping MQTT communications
broker_address = "test.mosquitto.org"
# In case "test.mosquitto.org" is not working, use the broker from HiveMQ mentioned below
#broker_address = "broker.hivemq.com"

client = mqtt.Client()

client.on_connect = on_connect
client.on_message = on_message

client.connect(broker_address,1883,60)

now = datetime.datetime.now()
time_stamp = str(now)

print("Current date and time: ", time_stamp)

client.loop_forever()

while True:
    client.subscribe("clock/detected")
    time.sleep(1)

##### Exercise

Use inference from an object detection model (image of apple) and communicate the bounding box coordinates using MQTT to topic name *apple/farm/season*

##### Solution

Use inference from an object detection model (image of apple) and communicate the bounding box coordinates using MQTT to topic name *apple/farm/season*

Solution:

- Start by installing Tflite runtime and MQTT Paho
- Perform inference on image using object detection model
- Store bounding box (inference output) as string
- Publish bounding box on MQTT topic *apple/farm/season*
- Subscribe to MQTT topic *apple/farm/season* to validate publish action

Do not run the cell below if the working directory contains the files downloaded earlier in the notebook

In [0]:
%%bash

wget -q https://edge-ai-doulos.s3.us-west-2.amazonaws.com/Edge-AI-Upload.zip
unzip -q Edge-AI-Upload.zip
pip install -q ai-edge-litert
pip install -q paho-mqtt==1.6.1

In [0]:
import time
import cv2
import paho.mqtt.client as mqtt  #import MQTT client functionality
from ai_edge_litert.interpreter import Interpreter
import numpy as np

# Initiliaze MQTT functions and broker address

def on_connect(client, userdata, flags, rc):
    print("MQTT connection received with result code " + str(rc))
    client.subscribe("apple/farm/season")


def on_message(client, userdata, message):
    print("MQTT message received ", str(message.payload.decode("utf-8")))
    print("Payload:", message.payload) # Print the payload


# Broker_address for prototyping MQTT communications
#broker_address = "test.mosquitto.org"
# In case "test.mosquitto.org" is not working, use the broker from HiveMQ mentioned below
broker_address = "broker.hivemq.com"

#Create new client instance
client = mqtt.Client()

#Attach connect to callback function
client.on_connect = on_connect

#Attach message received to callback function
client.on_message = on_message

#Connect to broker. Broker can be located on edge or cloud
client.connect(broker_address)


label2string = \
{
    0:   "person",
    1:   "bicycle",
    2:   "car",
    3:   "motorcycle",
    4:   "airplane",
    5:   "bus",
    6:   "train",
    7:   "truck",
    8:   "boat",
    9:   "traffic light",
    10:  "fire hydrant",
    12:  "stop sign",
    13:  "parking meter",
    14:  "bench",
    15:  "bird",
    16:  "cat",
    17:  "dog",
    18:  "horse",
    19:  "sheep",
    20:  "cow",
    21:  "elephant",
    22:  "bear",
    23:  "zebra",
    24:  "giraffe",
    26:  "backpack",
    27:  "umbrella",
    30:  "handbag",
    31:  "tie",
    32:  "suitcase",
    33:  "frisbee",
    34:  "skis",
    35:  "snowboard",
    36:  "sports ball",
    37:  "kite",
    38:  "baseball bat",
    39:  "baseball glove",
    40:  "skateboard",
    41:  "surfboard",
    42:  "tennis racket",
    43:  "bottle",
    45:  "wine glass",
    46:  "cup",
    47:  "fork",
    48:  "knife",
    49:  "spoon",
    50:  "bowl",
    51:  "banana",
    52:  "apple",
    53:  "sandwich",
    54:  "orange",
    55:  "broccoli",
    56:  "carrot",
    57:  "hot dog",
    58:  "pizza",
    59:  "donut",
    60:  "cake",
    61:  "chair",
    62:  "couch",
    63:  "potted plant",
    64:  "bed",
    66:  "dining table",
    69:  "toilet",
    71:  "tv",
    72:  "laptop",
    73:  "mouse",
    74:  "remote",
    75:  "keyboard",
    76:  "cell phone",
    77:  "microwave",
    78:  "oven",
    79:  "toaster",
    80:  "sink",
    81:  "refrigerator",
    83:  "book",
    84:  "clock",
    85:  "vase",
    86:  "scissors",
    87:  "teddy bear",
88: "hair drier",
    89:  "toothbrush",
}

def detect_from_image():
    # prepare input image
    start = time.time()
    img_org = cv2.imread('Images/apple.jpeg')
    img = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (300, 300))
    img = img.reshape(1, img.shape[0], img.shape[1],
                      img.shape[2])  # (1, 300, 300, 3)
    img = img.astype(np.uint8)

    # Overview of Object Detection: https://www.tensorflow.org/lite/examples/object_detection/overview
    # Load pretrained model:https://tfhub.dev/tensorflow/lite-model/ssd_mobilenet_v1/1/default/1

    interpreter = Interpreter(
        model_path="Model-Files/ssd_mobilenet_v1.tflite")

    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # set input tensor
    interpreter.set_tensor(input_details[0]['index'], img)

    # run
    interpreter.invoke()

    # get output tensor
    boxes = interpreter.get_tensor(output_details[0]['index'])
    boxes_shape = output_details[0]['shape_signature']
    labels = interpreter.get_tensor(output_details[1]['index'])
    scores = interpreter.get_tensor(output_details[2]['index'])
    num = interpreter.get_tensor(output_details[3]['index'])
    labels_list = labels.tolist()

    # Convert boxes to a NumPy array with a suitable data type
    boxes = np.array(boxes, dtype=np.float32)

    stop = time.time()
    print(f'time for inference is {stop-start:.2f} seconds')
    boxes_str = str(boxes) # Convert bounding box to string
    return boxes_str

# Publish inference to MQTT topic '/apple/farm/season'

def publish_inference():
    bounding_box = detect_from_image()
    print("Payload being sent:", bounding_box)
    client.publish("apple/farm/season", payload=bounding_box)
    print ('Inference published to MQTT topic')

def subscribe_inference():
   print ('Inference from MQTT topic')
   client.subscribe("apple/farm/season")

if __name__ == '__main__':
    client.loop_start()
    publish_inference()
    subscribe_inference()
    time.sleep(2)
    client.loop_stop()
    client.disconnect()
   